In [1]:
%cd ..

/Users/niccolotogni/Repo/prompt-forge


In [2]:
import logging
import os

from openai import AzureOpenAI
from dotenv import load_dotenv

from prompt_forge import (
    Project,
    LLMMessage,
    LLMResponse,
)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)


# ── Azure OpenAI client ───────────────────────────────────────────────────────
class AzureClient:
    ENDPOINT = "https://sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com/"
    API_VERSION = "2024-12-01-preview"
    DEPLOYMENT = "gpt-5.3-chat"  # ← change to your deployment name

    def __init__(self):
        self.client = AzureOpenAI(
            api_version=self.API_VERSION,
            azure_endpoint=self.ENDPOINT,
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
        )

    def complete(self, messages: list[LLMMessage], **kwargs) -> LLMResponse:
        # Strip kwargs that AzureOpenAI doesn't accept (e.g. custom keys)
        allowed = {"temperature", "max_tokens", "top_p", "frequency_penalty", "presence_penalty"}
        oai_kwargs = {k: v for k, v in kwargs.items() if k in allowed}

        resp = self.client.chat.completions.create(
            model=self.DEPLOYMENT,
            messages=[{"role": m.role, "content": m.content} for m in messages],
            **oai_kwargs,
        )
        return LLMResponse(
            text=resp.choices[0].message.content,
            usage={
                "input_tokens": resp.usage.prompt_tokens,
                "output_tokens": resp.usage.completion_tokens,
            },
        )


In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = "examples/extraction_example"  # Directory containing example subfolders (001/, 002/, ...)
PROJECT_DIR = ".projects/extraction_example"       # Directory to store project files (prompt versions, logs, etc.)

BUNDLE_SCHEMA = {
    "input": ".txt",
    "expected_output": ".json",
}

SEED_PROMPT = (
    "You are a helpful assistant. Process the input and produce the expected output."
)

CONTEXT = (
    "Debug task: echo or transform text inputs."
)

# Set to "none" to skip evaluation (faster iteration), or use
# "similarity", "exact_match", "json_fields", "llm_judge"
EVAL_STRATEGY = "json_fields"


In [4]:
load_dotenv()
llm = AzureClient()
project = Project(
    name="debug_project",
    llm=llm,
    project_dir=PROJECT_DIR,
)

project.set_bundle_schema(**BUNDLE_SCHEMA)
project.set_context(CONTEXT)
project.set_seed_prompt(SEED_PROMPT)

n = project.add_examples_from_directory(DATA_DIR)
print(f"\nLoaded {n} examples")

if n == 0:
    print(
        "\nNo examples found. Create them under:\n"
        f"  {DATA_DIR}/001/input.txt\n"
        f"  {DATA_DIR}/001/expected_output.txt\n"
        "Then re-run."
    )


2026-03-09 16:49:30,582 [INFO] prompt_forge.project: Bundle schema set: {'input': '.txt', 'expected_output': '.json'}
2026-03-09 16:49:30,583 [INFO] prompt_forge.project: Seed prompt set.
2026-03-09 16:49:30,583 [INFO] prompt_forge.bundle: Loading bundles from subdirectories in examples/extraction_example...
2026-03-09 16:49:30,586 [INFO] prompt_forge.project: Loaded 20 example bundles from examples/extraction_example



Loaded 20 examples


In [5]:
from prompt_forge.training.pipeline import TrainingConfig

train_config = TrainingConfig(
    batch_size=10,
    max_iterations=2,
    inference_temperature=1
)
print(f"\nStarting training on {project.num_examples} examples...")

report = project.train(
    config=train_config,
    eval_strategy=EVAL_STRATEGY,
)

print("\n── Training Report ─────────────────────────────────────────")
print(f"Final version : v{report.final_version}")
print(f"Final score   : {report.final_score}")
print(f"Total tokens  : {report.total_tokens_used:,}")
print(f"Refinement rec: {report.refinement_recommended}")
print()


2026-03-09 16:49:31,523 [INFO] prompt_forge.training.pipeline: Starting training: 20 examples, batch_size=10, max_iterations=2
2026-03-09 16:49:31,523 [INFO] prompt_forge.training.pipeline: 
Iteration 1/2
2026-03-09 16:49:31,524 [INFO] prompt_forge.training.pipeline: Selected batch: ['spec_03', 'spec_13', 'spec_05', 'spec_19', 'spec_20', 'spec_18', 'spec_12', 'spec_08', 'spec_07', 'spec_11']
2026-03-09 16:49:31,642 [DEBUG] httpcore.connection: connect_tcp.started host='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' port=443 local_address=None timeout=5.0 socket_options=None



Starting training on 20 examples...


2026-03-09 16:49:31,766 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x112042ba0>
2026-03-09 16:49:31,766 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x1119483b0> server_hostname='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' timeout=5.0
2026-03-09 16:49:31,990 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x112054410>
2026-03-09 16:49:31,991 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'POST']>
2026-03-09 16:49:31,992 [DEBUG] httpcore.http11: send_request_headers.complete
2026-03-09 16:49:31,992 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'POST']>
2026-03-09 16:49:31,993 [DEBUG] httpcore.http11: send_request_body.complete
2026-03-09 16:49:31,993 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'POST']>
2026-03-09 16:49:40,437 [DEBUG] h


── Training Report ─────────────────────────────────────────
Final version : v2
Final score   : 0.978
Total tokens  : 184,697
Refinement rec: False



In [6]:
for r in report:
    status = "✓" if r.improved else "✗"
    score_str = (
        f"{r.score_before:.3f} → {r.score_after:.3f}"
        if r.score_before is not None and r.score_after is not None
        else "no eval"
    )
    tokens_str = f"  [{r.tokens_used:,} tok]" if r.tokens_used else ""
    print(f"  Iter {r.iteration}: {score_str} {status}{tokens_str}")
    print(f"    Learned: {r.learnings[:120]}")

print("\n── Prompt versions ─────────────────────────────────────────")
for v in project.list_versions():
    score = f"score={v.eval_score:.3f}" if v.eval_score is not None else "no score"
    print(f"  v{v.version}  {score}  {(v.training_log_entry or '')[:80]}")

print("\n── Inference test ──────────────────────────────────────────")
agent = project.get_inference_agent()
print(agent.prompt_info)
result = agent.run(input_text="Hello, this is a test input.")
print(f"Output: {result}")


2026-03-09 17:07:07,027 [INFO] prompt_forge.inference.agent: Loaded prompt version 2 (score: 0.978)
2026-03-09 17:07:07,028 [DEBUG] httpcore.connection: close.started
2026-03-09 17:07:07,029 [DEBUG] httpcore.connection: close.complete
2026-03-09 17:07:07,029 [DEBUG] httpcore.connection: connect_tcp.started host='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' port=443 local_address=None timeout=5.0 socket_options=None
2026-03-09 17:07:07,150 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x112056710>
2026-03-09 17:07:07,150 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x1119483b0> server_hostname='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' timeout=5.0


  Iter 1: 0.042 → 0.922 ✓  [56,113 tok]
    Learned: - Product names must exclude descriptive machine-type suffixes (e.g., remove "Precision Surface Grinder").
- Many specif
  Iter 2: 0.914 → 0.978 ✓  [69,370 tok]
    Learned: - Unit conversions must be rounded consistently: dimensions and weight to whole integers, and Fahrenheit→Celsius tempera

── Prompt versions ─────────────────────────────────────────
  v0  no score  Seed prompt
  v1  score=0.922  - Product names must exclude descriptive machine-type suffixes (e.g., remove "Pr
  v2  score=0.978  - Unit conversions must be rounded consistently: dimensions and weight to whole 

── Inference test ──────────────────────────────────────────
Prompt: 272 lines, 8197 chars, output_schema=['product_name', 'manufacturer', 'part_number', 'year_of_manufacture', 'machine_type', 'application_sector', 'power_supply', 'dimensions', 'weight_kg', 'operating_conditions', 'performance', 'safety_certifications', 'communication_interfaces', 'maintenanc

2026-03-09 17:07:07,416 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x112184e90>
2026-03-09 17:07:07,417 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'POST']>
2026-03-09 17:07:07,420 [DEBUG] httpcore.http11: send_request_headers.complete
2026-03-09 17:07:07,421 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'POST']>
2026-03-09 17:07:07,423 [DEBUG] httpcore.http11: send_request_body.complete
2026-03-09 17:07:07,423 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'POST']>
2026-03-09 17:07:13,037 [DEBUG] httpcore.http11: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Length', b'2072'), (b'Content-Type', b'application/json'), (b'apim-request-id', b'e8fa6750-fe88-4e4b-b5c1-5b0d4e16e501'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'x-content-type-options', b'nosniff'), (b'x

Output: {'product_name': None, 'manufacturer': None, 'part_number': None, 'year_of_manufacture': None, 'machine_type': None, 'application_sector': None, 'power_supply': {'voltage_v': None, 'phases': None, 'frequency_hz': None, 'power_rating_kw': None}, 'dimensions': {'length_mm': None, 'width_mm': None, 'height_mm': None}, 'weight_kg': None, 'operating_conditions': {'temperature_min_c': None, 'temperature_max_c': None, 'humidity_max_pct': None}, 'performance': {'max_speed_rpm': None, 'throughput': None, 'accuracy_mm': None, 'noise_level_db': None}, 'safety_certifications': [], 'communication_interfaces': [], 'maintenance_interval_hours': None, 'warranty_years': None}


In [8]:
print(project.list_versions()[2].prompt_text)

You are an information extraction system that converts unstructured industrial equipment specification text into a structured JSON object.

Your task is to read the specification text and extract normalized structured data that strictly follows the schema below.

---------------------------------------------------------------------
CRITICAL OUTPUT RULES (HIGHEST PRIORITY)
---------------------------------------------------------------------

1. You MUST output ONLY a single valid JSON object.
2. The response MUST start with "{" and end with "}".
3. Do NOT include:
   - explanations
   - comments
   - markdown
   - code fences
   - notes
   - any text before or after the JSON
4. Use only valid JSON syntax:
   - double quotes for all keys and strings
   - no trailing commas
   - numbers must not be quoted
   - null must be the literal null (not "null")
5. Every field defined in the schema MUST appear in the output.
6. If a value cannot be determined, set it to null.
7. Arrays must always

In [11]:
report

TrainingReport(iterations=[IterationResult(iteration=1, prompt_version=2, score_before=0.0, score_after=0.0, improved=True, learnings='- Invalid JSON errors likely come from formatting issues; strict rules were added requiring the output to start with `{` and end with `}`, forbid code fences, and prohibit trailing commas.\n- Explicit escaping rules were needed for strings containing quotes (e.g., `19" rack` → `19\\" rack`).\n- Numeric normalization clarified for thresholds (`<`, `>`, `±`), percentages, and unit stripping to ensure numeric JSON types.\n- Added a mandatory validation checklist to enforce syntactic JSON correctness before output.', issues='- The schema is extremely large but expected outputs only include a subset of fields; examples imply unused fields should be omitted rather than always set to null, but this rule is not explicitly defined.\n- Acronym extraction expectations may be ambiguous (some outputs split terms like "OPC UA" into "OPC" and "UA" while others keep co